# ECG Arrhythmia Detection — CNN-BiLSTM
**Dataset**: MIT-BIH Arrhythmia Database (PhysioNet)

**Task**: Multi-class beat classification (N, L, R, V, A)

**Stack**: PyTorch, MLflow, scikit-learn

> Before running: Runtime -> Change runtime type -> T4 GPU

## 1. Install dependencies

In [ ]:
!pip install wfdb mlflow imbalanced-learn scikit-learn scipy seaborn -q

## 2. Imports

In [ ]:
import os
import pickle
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from scipy.signal import butter, filtfilt
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


## 3. Configuration

In [ ]:
CFG = {
    'window_size': 180,
    'batch_size': 256,
    'epochs': 50,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'cnn_filters': [32, 64, 128],
    'kernel_size': 5,
    'lstm_hidden': 128,
    'lstm_layers': 2,
    'dropout': 0.3,
    'num_classes': 5,
    'label_map': {
        'N': 'N', 'L': 'L', 'R': 'R',
        'V': 'V', '/': 'V',
        'A': 'A', 'a': 'A',
        'e': 'N', 'j': 'N', 'n': 'N',
        'E': 'V', 'F': 'V',
        'f': 'N', 'Q': 'N', '!': 'V'
    },
    'classes': ['N', 'L', 'R', 'V', 'A'],
    'mlflow_experiment': 'ecg-arrhythmia-detection'
}

## 4. Download MIT-BIH dataset

In [ ]:
RECORD_IDS = [
    '100','101','102','103','104','105','106','107',
    '108','109','111','112','113','114','115','116',
    '117','118','119','121','122','123','124','200',
    '201','202','203','205','207','208','209','210',
    '212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Downloading {len(RECORD_IDS)} records...')
for rid in RECORD_IDS:
    try:
        wfdb.dl_database('mitdb', dl_dir=DATA_DIR, records=[rid])
    except Exception as e:
        print(f'  skipping {rid}: {e}')

print('Download complete.')

## 5. Signal preprocessing and beat segmentation

In [ ]:
def bandpass_filter(signal, lowcut=0.5, highcut=50.0, fs=360, order=4):
    # remove baseline wander and high-frequency noise
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, signal)


def normalize_segment(segment):
    # z-score normalize each beat independently
    std = np.std(segment)
    return (segment - np.mean(segment)) / (std if std > 0 else 1.0)


def extract_beats(record_path, label_map, window=180):
    # extract fixed-length windows centered on each annotated R-peak
    try:
        record = wfdb.rdrecord(record_path)
        annotation = wfdb.rdann(record_path, 'atr')
    except Exception:
        return [], []

    ecg = record.p_signal[:, 0].astype(np.float32)
    ecg = bandpass_filter(ecg)
    half = window // 2
    beats, labels = [], []

    for sample_idx, symbol in zip(annotation.sample, annotation.symbol):
        if symbol not in label_map:
            continue
        start = sample_idx - half
        end = sample_idx + half
        if start < 0 or end > len(ecg):
            continue
        beats.append(normalize_segment(ecg[start:end]))
        labels.append(label_map[symbol])

    return beats, labels


all_beats, all_labels = [], []
for rid in RECORD_IDS:
    b, l = extract_beats(os.path.join(DATA_DIR, rid), CFG['label_map'], CFG['window_size'])
    all_beats.extend(b)
    all_labels.extend(l)

X = np.array(all_beats, dtype=np.float32)
y_raw = np.array(all_labels)

# keep only the 5 target classes
mask = np.isin(y_raw, CFG['classes'])
X, y_raw = X[mask], y_raw[mask]

le = LabelEncoder()
le.fit(CFG['classes'])
y = le.transform(y_raw)

print(f'Total beats: {len(X)}')
for cls, cnt in zip(*np.unique(y_raw, return_counts=True)):
    print(f'  {cls}: {cnt}')

## 6. Train/val/test split and SMOTE oversampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

# apply SMOTE only to training data to avoid leakage
smote = SMOTE(random_state=SEED)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print('Training class distribution after SMOTE:')
for cls_idx, cnt in zip(*np.unique(y_train, return_counts=True)):
    print(f'  {le.classes_[cls_idx]}: {cnt}')

## 7. PyTorch Dataset and DataLoaders

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        # unsqueeze adds channel dim: (N, 1, window_size) required by Conv1d
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# use for GPU
# train_loader = DataLoader(ECGDataset(X_train, y_train), batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
# val_loader   = DataLoader(ECGDataset(X_val,   y_val),   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
# test_loader  = DataLoader(ECGDataset(X_test,  y_test),  batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# Use for MPS
train_loader = DataLoader(ECGDataset(X_train, y_train), batch_size=CFG['batch_size'], shuffle=True)
val_loader   = DataLoader(ECGDataset(X_val,   y_val),   batch_size=CFG['batch_size'], shuffle=False)
test_loader  = DataLoader(ECGDataset(X_test,  y_test),  batch_size=CFG['batch_size'], shuffle=False)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 8. CNN-BiLSTM model

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # CNN blocks extract local morphological features from the waveform
        cnn_layers = []
        in_ch = 1
        for out_ch in cfg['cnn_filters']:
            cnn_layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=cfg['kernel_size'], padding=cfg['kernel_size'] // 2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Dropout(cfg['dropout'] * 0.5),
            ]
            in_ch = out_ch
        self.cnn = nn.Sequential(*cnn_layers)

        # BiLSTM captures temporal dependencies across the beat sequence
        self.bilstm = nn.LSTM(
            input_size=cfg['cnn_filters'][-1],
            hidden_size=cfg['lstm_hidden'],
            num_layers=cfg['lstm_layers'],
            batch_first=True,
            bidirectional=True,
            dropout=cfg['dropout'] if cfg['lstm_layers'] > 1 else 0.0,
        )

        # attention layer weights which timesteps matter most
        self.attention = nn.Linear(cfg['lstm_hidden'] * 2, 1)

        # final classification head
        self.classifier = nn.Sequential(
            nn.Linear(cfg['lstm_hidden'] * 2, 64),
            nn.ReLU(),
            nn.Dropout(cfg['dropout']),
            nn.Linear(64, cfg['num_classes']),
        )

    def forward(self, x):
        x = self.cnn(x)                          # (batch, 128, seq_len)
        x = x.permute(0, 2, 1)                   # (batch, seq_len, 128)
        lstm_out, _ = self.bilstm(x)             # (batch, seq_len, hidden*2)
        attn = torch.softmax(self.attention(lstm_out), dim=1)
        context = (lstm_out * attn).sum(dim=1)   # (batch, hidden*2)
        return self.classifier(context)


model = CNNBiLSTM(CFG).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print(model)

## 9. Training and evaluation functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            preds = logits.argmax(1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    return total_loss / total, correct / total, macro_f1, all_preds, all_targets

## 10. Training loop with MLflow tracking

In [ ]:
mlflow.set_experiment(CFG['mlflow_experiment'])

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)

best_val_f1 = 0.0
best_model_path = 'best_model.pt'
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

with mlflow.start_run(run_name='cnn-bilstm-v1'):
    mlflow.log_params({
        'window_size':  CFG['window_size'],
        'batch_size':   CFG['batch_size'],
        'epochs':       CFG['epochs'],
        'lr':           CFG['lr'],
        'weight_decay': CFG['weight_decay'],
        'lstm_hidden':  CFG['lstm_hidden'],
        'lstm_layers':  CFG['lstm_layers'],
        'dropout':      CFG['dropout'],
        'cnn_filters':  str(CFG['cnn_filters']),
        'optimizer':    'Adam',
        'scheduler':    'ReduceLROnPlateau',
    })

    for epoch in range(1, CFG['epochs'] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        mlflow.log_metrics({
            'train_loss':   train_loss,
            'val_loss':     val_loss,
            'train_acc':    train_acc,
            'val_acc':      val_acc,
            'val_macro_f1': val_f1,
        }, step=epoch)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)

        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}/{CFG["epochs"]} | '
                  f'train loss: {train_loss:.4f} acc: {train_acc:.4f} | '
                  f'val loss: {val_loss:.4f} acc: {val_acc:.4f} f1: {val_f1:.4f}')

    # load best checkpoint and run final test evaluation
    model.load_state_dict(torch.load(best_model_path))
    test_loss, test_acc, test_f1, test_preds, test_targets = evaluate(model, test_loader, criterion)

    mlflow.log_metrics({'test_loss': test_loss, 'test_acc': test_acc, 'test_macro_f1': test_f1})
    mlflow.pytorch.log_model(model, artifact_path='model')
    mlflow.log_artifact(best_model_path)

    print(f'\nBest val F1:   {best_val_f1:.4f}')
    print(f'Test accuracy: {test_acc:.4f}')
    print(f'Test macro F1: {test_f1:.4f}')

## 11. Training curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, history['train_loss'], label='train')
axes[0].plot(epochs_range, history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='train')
axes[1].plot(epochs_range, history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, history['val_f1'], color='green', label='val macro F1')
axes[2].set_title('Validation macro F1'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Classification report and confusion matrix

In [ ]:
class_names = le.classes_.tolist()
print(classification_report(test_targets, test_preds, target_names=class_names))

cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[0].set_title('Confusion matrix (counts)');     axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
axes[1].set_title('Confusion matrix (normalized)'); axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Sample beat predictions

In [ ]:
model.eval()
X_sample, y_sample = next(iter(test_loader))
with torch.no_grad():
    logits = model(X_sample.to(DEVICE))
    probs  = torch.softmax(logits, dim=1).cpu().numpy()
    preds  = logits.argmax(1).cpu().numpy()

X_np = X_sample.squeeze(1).numpy()
y_np = y_sample.numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes.flatten()):
    true_cls = class_names[y_np[i]]
    pred_cls = class_names[preds[i]]
    color = 'green' if true_cls == pred_cls else 'red'
    ax.plot(X_np[i], linewidth=0.8, color='steelblue')
    ax.set_title(f'True: {true_cls}  Pred: {pred_cls} ({probs[i][preds[i]]*100:.1f}%)', color=color, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.suptitle('Sample beat predictions (green = correct, red = wrong)', fontsize=11)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Rhythm strip visualization (hypnogram-style)

In [ ]:
DEMO_RECORD = '208'  # record 208 has rich arrhythmia variety
beats, labels = extract_beats(os.path.join(DATA_DIR, DEMO_RECORD), CFG['label_map'], CFG['window_size'])

if beats:
    X_demo = torch.tensor(np.array(beats, dtype=np.float32)).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        logits_demo = model(X_demo)
        preds_demo  = logits_demo.argmax(1).cpu().numpy()
        probs_demo  = torch.softmax(logits_demo, dim=1).cpu().numpy()

    true_numeric = np.array([le.transform([l])[0] if l in le.classes_ else -1 for l in labels])
    valid = true_numeric >= 0
    pred_seq  = preds_demo[valid]
    true_seq  = true_numeric[valid]
    time_axis = np.arange(valid.sum())

    colors = {'N': '#2196F3', 'L': '#4CAF50', 'R': '#FF9800', 'V': '#F44336', 'A': '#9C27B0'}

    fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)

    for t, p in zip(time_axis, pred_seq):
        axes[0].axvspan(t, t+1, color=colors[class_names[p]], alpha=0.8)
    axes[0].set_ylabel('Predicted'); axes[0].set_yticks([])

    for t, tr in zip(time_axis, true_seq):
        axes[1].axvspan(t, t+1, color=colors[class_names[tr]], alpha=0.8)
    axes[1].set_ylabel('True'); axes[1].set_yticks([])

    conf = probs_demo[valid].max(axis=1)
    axes[2].plot(time_axis, conf, color='gray', linewidth=0.6)
    axes[2].fill_between(time_axis, conf, alpha=0.3, color='gray')
    axes[2].set_ylabel('Confidence'); axes[2].set_ylim(0, 1); axes[2].set_xlabel('Beat index')

    legend_handles = [Patch(color=v, label=k) for k, v in colors.items()]
    fig.legend(handles=legend_handles, loc='upper right', ncol=5, fontsize=9)
    plt.suptitle(f'Rhythm strip — record {DEMO_RECORD}', fontsize=12)
    plt.tight_layout()
    plt.savefig('rhythm_strip.png', dpi=150, bbox_inches='tight')
    plt.show()

## 15. Save artifacts for inference service

In [ ]:
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

with open('model_config.json', 'w') as f:
    json.dump(CFG, f, indent=2)

print('Saved: best_model.pt, label_encoder.pkl, model_config.json')
print('These three files are all you need to rebuild the inference service.')

## 16. Download all artifacts from Colab

In [ ]:
from google.colab import files

for fname in [
    'best_model.pt', 'label_encoder.pkl', 'model_config.json',
    'training_curves.png', 'confusion_matrix.png',
    'sample_predictions.png', 'rhythm_strip.png'
]:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')

## 17. View MLflow UI in Colab (optional)

In [ ]:
!pip install pyngrok -q
import subprocess, time
from pyngrok import ngrok

# start MLflow server in background
subprocess.Popen(['mlflow', 'ui', '--port', '5000'])
time.sleep(2)

# open a public tunnel so the UI is accessible from your browser
public_url = ngrok.connect(5000)
print(f'MLflow UI: {public_url}')
print('Open the link above to explore your experiment runs and screenshots.')